<a href="https://colab.research.google.com/github/springboardmentor054-coder/-VisionExtract-Isolation-from-Images-using-Image-Segmentation/blob/main/Milestone_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install gradio torch torchvision opencv-python pillow matplotlib

In [1]:
import torch
from torchvision import models
import torchvision.transforms as T

device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.segmentation.deeplabv3_resnet101(pretrained=True)
model.to(device)
model.eval()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/deeplabv3_resnet101_coco-586e9e4e.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet101_coco-586e9e4e.pth


100%|██████████| 233M/233M [00:02<00:00, 96.4MB/s]


DeepLabV3(
  (backbone): IntermediateLayerGetter(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Se

In [2]:
transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224,224)),
    T.ToTensor()
])

In [3]:
import numpy as np
import cv2
from PIL import Image

def process_image(image):
    original = np.array(image)

    # Preprocess
    input_tensor = transform(original).unsqueeze(0).to(device)

    # Prediction
    with torch.no_grad():
        output = model(input_tensor)['out'][0]

    # Get mask (person class = 15 in COCO)
    mask = output.argmax(0).cpu().numpy()
    binary_mask = (mask == 15).astype(np.uint8)

    # Resize mask back
    binary_mask = cv2.resize(binary_mask, (original.shape[1], original.shape[0]))

    # Apply mask
    result = original * np.expand_dims(binary_mask, axis=2)

    return (
        Image.fromarray(original),
        Image.fromarray(binary_mask * 255),
        Image.fromarray(result)
    )

In [4]:
import gradio as gr

def app(image):
    before, mask, after = process_image(image)
    return before, mask, after

demo = gr.Interface(
    fn=app,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Original Image"),
        gr.Image(label="Predicted Mask"),
        gr.Image(label="Isolated Subject")
    ],
    title="VisionExtract - AI Subject Isolation",
    description="Upload an image → get subject extracted"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bbba346eb8d5a237dd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
# =========================
# INSTALL
# =========================
!pip install gradio torch torchvision pillow opencv-python

# =========================
# IMPORTS
# =========================
import torch
from torchvision import models
import torchvision.transforms as T
import numpy as np
import cv2
from PIL import Image
import gradio as gr

# =========================
# LOAD MODEL (Pretrained DeepLabV3)
# =========================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = models.segmentation.deeplabv3_resnet101(pretrained=True)
model.to(device)
model.eval()

# =========================
# TRANSFORM
# =========================
transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224,224)),
    T.ToTensor()
])

# =========================
# SINGLE IMAGE PROCESSING
# =========================
def process_image(image):
    original = np.array(image)

    input_tensor = transform(original).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(input_tensor)['out'][0]

    mask = output.argmax(0).cpu().numpy()

    # Extract PERSON (class 15)
    binary_mask = (mask == 15).astype(np.uint8)

    # Resize mask to original size
    binary_mask = cv2.resize(binary_mask, (original.shape[1], original.shape[0]))

    result = original * np.expand_dims(binary_mask, axis=2)

    return (
        Image.fromarray(original),
        Image.fromarray(binary_mask * 255),
        Image.fromarray(result)
    )

# =========================
# MULTIPLE IMAGE PROCESSING
# =========================
def process_multiple(files):
    if len(files) > 3:
        return ["⚠️ Upload maximum 3 images only"]

    gallery = []

    for file in files:
        image = Image.open(file.name).convert("RGB")

        before, mask, after = process_image(image)

        # Add all 3 outputs
        gallery.append(before)
        gallery.append(mask)
        gallery.append(after)

    return gallery

# =========================
# GRADIO UI
# =========================
demo = gr.Interface(
    fn=process_multiple,
    inputs=gr.File(file_count="multiple"),
    outputs=gr.Gallery(label="Results (Before | Mask | After)"),
    title="VisionExtract - Multi Image Subject Isolation",
    description="Upload up to 3 images → get segmentation results"
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet101_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1de47e87211d06822a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
